<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/965_SEv3_Metrics_Spec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 METRICS_SPEC_SE_v3.md

### *Sales Enablement Orchestrator — v3 Metrics Framework*

---

# Purpose

This document defines the **official metrics, calculations, and trigger logic** for the Sales Enablement Orchestrator v3.

The goal is not to produce dashboards.

The goal is:

> **Deterministic, explainable, executive-ready revenue intelligence**

This framework enables:

* deal-level risk detection
* pipeline trajectory analysis
* rep performance evaluation
* forward-looking revenue signals

---

# 🧭 Metric Architecture

The system evaluates performance at **three levels**:

```
Deal → Rep → Pipeline
```

Each level includes:

* current state (snapshot)
* historical trend (v3 requirement)
* risk velocity (v3 requirement)
* executive triggers

---

# 1️⃣ DEAL-LEVEL METRICS (Primary Unit)
# 📦 Data Contract (Field-to-Metric Mapping)

## Source datasets

* `agents/data/deals.json` provides deal identity + current snapshot fields (including `deal_id`, `lead_id`, `rep_id`, `created_date`, `stage`, `deal_value_usd`, `probability`, `risk_flags`, `status`).
* `agents/data/deal_history.json` provides the temporal trajectory per deal:
  * `deal_id`
  * `monthly_history[]` with `period` (`YYYY-MM`), `stage`, `probability`, `deal_value_usd`
* `agents/data/interactions.json` provides interaction timestamps:
  * `deal_id`, `datetime` (ISO, `...Z`)
* `agents/data/signals.json` provides deterministic signals at the lead level:
  * `lead_id -> engagement_score`, `deal_risk_score`
  * Mapping rule: for a deal, use `signals[deal.lead_id]`.
* `agents/data/rep_performance_history.json` provides rep temporal performance:
  * `rep_id`, `monthly_history[]` with `period`, `close_rate`, `pipeline_value`
* `agents/data/pipeline_snapshots.json` provides pipeline temporal performance:
  * `period`, `total_pipeline_value`, `weighted_pipeline_value`, `win_rate`
* `agents/data/stage_actions.json` provides stage expectations used for velocity:
  * `stage -> if_stalled_days` (expected days in stage)
* `agents/data/thresholds.json` provides deterministic thresholds.

## Threshold keys used in v3

* `stalled_deal_days_in_stage` (default: `21`)
* `high_value_deal_min_usd` (default: `100000`)
* `min_interaction_recency_days` (default: `7`)
* `pipeline_coverage_ratio_min` (default: `0.8`)

## 1.1 Deal Momentum Score

**Purpose:** Detect whether a deal is progressing or stalling.

### Inputs:

* interaction frequency (interactions.json)
* stage changes (deal_history.json)
* recency (last interaction)

### Logic (deterministic):

Definitions:

* `stage_changed_recently` is true when the latest `deal_history.monthly_history` point has `stage_current_period != stage_previous_period`.
* `interactions_last_14_days` counts interactions for the deal where:
  * `period_end_date_utc - 14 days <= interaction.datetime_utc_date <= period_end_date_utc`
  * `period_end_date_utc` is the UTC month-end date for `monthly_history.period` (`YYYY-MM`).

```python
if interactions_last_14_days >= 2 and stage_changed_recently:
    momentum = "accelerating"
elif interactions_last_14_days == 0:
    momentum = "stalled"
else:
    momentum = "stable"
```

---

## 1.2 Probability Trend

**Purpose:** Detect confidence direction.

```python
probability_trend = current_probability - avg(previous_3_probabilities)
```

Implementation note:

* `previous_3_probabilities` is the average of up to the previous 3 probability points in the deal’s `deal_history` timeline.

### Output:

* positive → improving
* negative → deteriorating

---

## 1.3 Stage Velocity

**Purpose:** Measure pipeline efficiency.

```python
velocity_ratio = days_in_stage / expected_days_in_stage
```

Definitions:

* `days_in_stage` is computed per period as `(period_end_date_utc - deal.created_date_utc_date).days`, where `period_end_date_utc` is the UTC month-end for `deal_history.monthly_history.period`.
* `expected_days_in_stage` is derived from `stage_actions.json`:
  * `expected_days_in_stage = stage_actions[stage_current_period].if_stalled_days`
  * fallback: `thresholds.json["stalled_deal_days_in_stage"]`.

### Interpretation:

* < 1.0 → fast
* ~1.0 → normal
* \> 1.5 → slow / risk

---

## 1.4 Deal Risk Score (v3 Composite)

**Purpose:** Unified risk signal.

### Inputs:

* engagement_score (from `signals.json` via `signals[deal.lead_id].engagement_score`)
* deal_risk_score (from `signals.json` via `signals[deal.lead_id].deal_risk_score`)
* probability_trend
* velocity_ratio

Note:

* interaction recency (`interaction_gap`) is used for deal trigger detection (no-interaction trigger), not as a term in the composite `risk_score` formula.

### Example formula:

```python
risk_score = (
    (1 - engagement_score) * 0.3 +
    deal_risk_score * 0.3 +
    max(0, -probability_trend) * 0.2 +
    min(1, velocity_ratio / 2) * 0.2
)
```

---

## 1.5 Forecast Stability (v3)

**Purpose:** Detect volatility.

```python
stability = std_dev(probability_history)
```

Implementation note:

* `probability_history` is the `deal_history.monthly_history[].probability` sequence for the deal across the available timeline.

### Interpretation:

* high std dev → unstable forecast

---

## 1.6 Stagnation Flag

Triggered when:

```python
if days_in_stage > stalled_threshold:
    stagnation_flag = True
```

(uses thresholds.json)

Implementation note:

* `stalled_threshold = thresholds.json["stalled_deal_days_in_stage"]` and the trigger uses a strict `>` comparison.

---

# 2️⃣ REP-LEVEL METRICS (Execution Layer)

## 2.1 Close Rate Trend

```python
trend = current_close_rate - avg(previous_close_rates)
```

Implementation note:

* `previous_close_rates` is the average of up to the previous 3 periods in `rep_performance_history`.

---

## 2.2 Pipeline Productivity

```python
productivity = pipeline_value / active_deals
```

Implementation note:

* `active_deals` is counted from `deals.json` where `rep_id` matches and `status == "active"`.

---

## 2.3 Rep Performance Score

Composite:

```python
rep_score = (
    close_rate * 0.4 +
    productivity_normalized * 0.3 +
    pipeline_growth * 0.3
)
```

Implementation note (normalization):

* `productivity_normalized = productivity / max(productivity across reps in the run)`
* `pipeline_growth_normalized = max(0, pipeline_growth / max_positive_pipeline_growth across reps)`

---

## 2.4 Performance Volatility

```python
volatility = std_dev(close_rate_history)
```

---

## 2.5 Rep Risk Flag

Triggered when:

* close rate declines for 2 consecutive transitions (latest < previous < pre-previous)
* AND pipeline value decreased compared to the immediate previous period

---

# 3️⃣ PIPELINE-LEVEL METRICS (Executive Layer)

## 3.1 Pipeline Growth Rate

```python
growth_rate = (current_pipeline - previous_pipeline) / previous_pipeline
```

Implementation note:

* v3 uses `pipeline_snapshots.total_pipeline_value` for `current_pipeline` and `previous_pipeline`.

---

## 3.2 Weighted Pipeline Trend

Uses probability-adjusted revenue:

```python
weighted_pipeline = sum(deal_value * probability)
```

Implementation note:

* v3 uses `pipeline_snapshots.weighted_pipeline_value` directly.
* `weighted_pipeline_trend = weighted_pipeline_current - avg(weighted_pipeline_previous_up_to_3_periods)`.

Trend computed across periods.

---

## 3.3 Win Rate Trend

```python
win_rate_trend = current_win_rate - avg(previous_win_rates)
```

Implementation note:

* `previous_win_rates` is the average of up to the previous 3 periods in `pipeline_snapshots`.

---

## 3.4 Revenue Trajectory (🔥 Core v3 Metric)

**Purpose:** Direction of expected revenue.

### Derived from:

* weighted pipeline trend
* probability trend across deals

### Output:

* "increasing"
* "stable"
* "declining"

Implementation note:

* `probability trend across deals` is the average of each deal’s latest `probability_trend_current`.
* v3 sets:
  * `"increasing"` when `weighted_pipeline_trend > 0` AND `avg_deal_probability_trend > 0`
  * `"declining"` when both are `< 0`
  * `"stable"` otherwise

---

## 3.5 Pipeline Coverage Ratio

```python
coverage = pipeline_value / quota
```

Implementation note:

* `pipeline_value` is `pipeline_snapshots.total_pipeline_value` (latest period).
* `quota` is the sum of `quota_usd` across all reps in `sales_reps.json`.
* the executive threshold is `thresholds.json["pipeline_coverage_ratio_min"]` (default `0.8`).

---

# 4️⃣ RISK VELOCITY (v3 Core Layer)

## 4.1 Deal Risk Velocity

```python
risk_delta = current_risk - previous_risk
```

---

## 4.2 Velocity Classification

```python
if risk_delta > 0.1:
    velocity = "increasing"
elif risk_delta < -0.1:
    velocity = "decreasing"
else:
    velocity = "stable"
```

---

## 4.3 Portfolio Risk Velocity

v3 computes risk velocity at the deal level (`risk_delta` and `risk_velocity_class`) and emits trigger evolution per deal.

If you want a single portfolio-level scalar, a deterministic option is:
```python
portfolio_risk_velocity_scalar = avg(deal_risk_delta_current_across_deals)
```

This scalar is not required for executive triggers in the current implementation; the executive view is driven by deal/rep/pipeline trigger evolution metadata.

---

# 5️⃣ EXECUTIVE TRIGGERS (Non-Negotiable)

Triggers are **deterministic, auditable, and visible**.

---

## 5.1 Deal-Level Triggers

Trigger conditions (evaluated at the latest period in each deal’s history):

* `deal_high_value_deteriorating`: `deal_value_current >= thresholds.high_value_deal_min_usd` AND `probability_trend_current < 0.0`
* `deal_stalled_beyond_threshold`: `days_in_stage_current > thresholds.stalled_deal_days_in_stage`
* `no_interaction_in_x_days`: `interaction_gap_days_current > thresholds.min_interaction_recency_days`
* `risk_velocity_increasing`: `risk_delta_current > 0.1` (same threshold as velocity classification)

Definitions:

* `interaction_gap_days_current`: days between `period_end_date_utc` (UTC month-end for the deal’s latest `monthly_history.period`) and the most recent interaction datetime on/before that period end (from `interactions.json`).

---

## 5.2 Rep-Level Triggers

Trigger conditions (evaluated at each rep’s latest available period):

* `close_rate_declining_2plus`: close rate decreased across two consecutive transitions (latest < previous < pre-previous)
* `pipeline_shrinking`: pipeline value decreased across two consecutive transitions
* `high_value_deals_consistently_lost`: at least one high-value deal (`deal_value_usd >= thresholds.high_value_deal_min_usd`) was `Closed Lost` in both the current and previous periods for that rep (derived from `deal_history.json`)

---

## 5.3 Pipeline-Level Triggers

Trigger conditions:

* `pipeline_shrinking_2_periods`: `total_pipeline_value` decreased across two consecutive transitions
* `weighted_pipeline_declining`: `weighted_pipeline_value` decreased vs the immediate previous period
* `win_rate_declining`: `win_rate` decreased vs the immediate previous period
* `coverage_ratio_below_threshold`: `coverage_ratio_current < thresholds.pipeline_coverage_ratio_min`

---

## 5.4 Trigger Evolution (v3 Requirement)

Track:

```python
trigger_status_current
trigger_status_previous
trigger_duration_periods
new_trigger_flag
```

Implementation note:

* `new_trigger_flag` is true when the trigger is `True` in the latest period AND `False` in the immediately previous period.
* `trigger_duration_periods` counts consecutive `True` values ending at the latest period.

---

# 6️⃣ OUTPUT CLASSIFICATIONS

Every metric should resolve to:

* improving / stable / declining
* low / medium / high risk
* accelerating / stalled

---

# 7️⃣ DESIGN PRINCIPLES (Enforced)

### Deterministic

All metrics must be reproducible from data.

### Explainable

Every output must map to a formula.

### Minimal

No unnecessary metrics.

### Executive-Relevant

Every metric must answer:

> “What decision does this enable?”

---

# 🔥 Final Insight (Signature)

> **Revenue risk is not defined by current pipeline state.
> It is defined by trajectory, velocity, and execution quality.**



